# Job Market Analysis and Recommendation System

**Dataset:** `all_upwork_jobs_2024-02-07-2024-03-24.csv`

Run cells from top to bottom in Google Colab. This project analyzes job title keywords and rates, category demand, weekly forecasts, country differences, personalized recommendations, monthly changes, remote-work signals, and future trends.

**Important:** This dataset covers a short period and is from Upwork. Results describe this dataset, not the entire global job market. Currency is assumed to be USD only where Upwork rate fields are denominated in USD; verify the source before publication.

## 0. Setup and load data
Upload the CSV to Colab when prompted. If you put it in Google Drive instead, mount Drive and set `DATA_PATH` to the correct path.

In [ ]:
import io, re, warnings, os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

uploaded = files.upload()  # Select all_upwork_jobs_2024-02-07-2024-03-24.csv
DATA_PATH = next(iter(uploaded))
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
print(df.info())

## 1. Data quality and cleaning
Keep the original columns, parse dates, remove duplicate postings, and create analysis fields. Missing rates are not automatically zero.

In [ ]:
expected = ['title','link','published_date','is_hourly','hourly_low','hourly_high','budget','country']
missing_cols = sorted(set(expected) - set(df.columns))
if missing_cols:
    raise ValueError(f"Dataset is missing expected columns: {missing_cols}")

print("Missing values before cleaning:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate links:", df['link'].duplicated().sum())

df['published_date'] = pd.to_datetime(df['published_date'], errors='coerce', utc=True)
for col in ['hourly_low','hourly_high','budget']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['title'] = df['title'].fillna('').astype(str).str.strip()
df['country'] = df['country'].fillna('Unknown').astype(str).str.strip()
df = df.dropna(subset=['published_date']).drop_duplicates(subset=['link']).copy()
df = df[df['title'].ne('')].copy()
df['date'] = df['published_date'].dt.tz_convert(None)
df['week'] = df['published_date'].dt.tz_convert(None).dt.to_period('W').dt.start_time
df['month'] = df['published_date'].dt.tz_convert(None).dt.to_period('M').astype(str)
df['hourly_mid'] = df[['hourly_low','hourly_high']].mean(axis=1, skipna=True)
df['hourly_mid'] = df['hourly_mid'].where(df['is_hourly'] == True)
# Flag implausible/extreme values for sensitivity analysis; do not silently delete them.
df['hourly_rate_suspect'] = (df['hourly_mid'] <= 0) | (df['hourly_mid'] > 300)
df['budget_suspect'] = (df['budget'] <= 0) | (df['budget'] > 100000)
print("Cleaned shape:", df.shape)
print("Date range:", df['date'].min(), "to", df['date'].max())
display(df.isna().sum().to_frame('missing_count'))

## 2. Overview and exploratory data analysis

In [ ]:
print("Total postings:", len(df))
print("Hourly/fixed-price split:\n", df['is_hourly'].map({True:'Hourly',False:'Fixed-price'}).value_counts())
print("Top countries:\n", df['country'].value_counts().head(15))

fig, axes = plt.subplots(1, 2, figsize=(14,5))
df['is_hourly'].map({True:'Hourly',False:'Fixed-price'}).value_counts().plot(kind='bar', ax=axes[0], title='Posting type')
df['country'].value_counts().head(10).sort_values().plot(kind='barh', ax=axes[1], title='Top 10 countries')
plt.tight_layout(); plt.show()

weekly_total = df.groupby('week').size().rename('postings')
plt.figure(figsize=(14,5)); weekly_total.plot(marker='o')
plt.title('Job postings per week'); plt.xlabel('Week'); plt.ylabel('Postings'); plt.xticks(rotation=45); plt.tight_layout(); plt.show()
print("Weekly posting counts:\n", weekly_total.to_string())

## 3. Task 1 — Job title keywords versus offered hourly rates
Use the midpoint of low/high rates when available. This analysis is associative, not causal. Keywords with very small samples are excluded. Extreme rates are reported separately through a sensitivity filter.

In [ ]:
# Simple interpretable keyword dictionary; expand it for your domain if needed.
keyword_groups = {
 'Python':['python'], 'Data Science':['data scientist','data science','machine learning','ml engineer'],
 'Data Analytics':['data analyst','data analytics','business intelligence','power bi','tableau'],
 'AI / LLM':['artificial intelligence',' ai ','llm','generative ai','chatgpt','prompt engineer'],
 'Web Development':['web developer','frontend','front-end','backend','back-end','full stack','react','node.js'],
 'Mobile Development':['android','ios developer','flutter','react native'],
 'Design':['graphic design','ui/ux','ui designer','ux designer','figma'],
 'Writing':['writer','copywriter','content writing','blog writer'],
 'Marketing':['digital marketing','seo','social media','media buyer'],
 'Virtual Assistance':['virtual assistant','data entry','administrative assistant'],
 'Cloud / DevOps':['aws','azure','devops','kubernetes','docker'],
 'Cybersecurity':['cybersecurity','penetration testing','security analyst']
}
def assign_categories(title):
    t = ' ' + str(title).lower() + ' '
    found = [cat for cat, kws in keyword_groups.items() if any(k in t for k in kws)]
    return found or ['Other / Unclassified']

df['categories'] = df['title'].apply(assign_categories)
exploded = df.explode('categories')
rate_data = exploded[(exploded['is_hourly']) & exploded['hourly_mid'].notna() & (~exploded['hourly_rate_suspect'])].copy()
keyword_salary = rate_data.groupby('categories').agg(
    postings_with_rates=('hourly_mid','size'), median_hourly_rate=('hourly_mid','median'),
    mean_hourly_rate=('hourly_mid','mean'), q25=('hourly_mid',lambda x:x.quantile(.25)),
    q75=('hourly_mid',lambda x:x.quantile(.75))).query('postings_with_rates >= 30').sort_values('median_hourly_rate',ascending=False)
display(keyword_salary.round(2))
plt.figure(figsize=(11,6)); keyword_salary.sort_values('median_hourly_rate').median_hourly_rate.plot(kind='barh')
plt.title('Median hourly rate by job-title category (minimum 30 rate records)'); plt.xlabel('Hourly rate'); plt.tight_layout(); plt.show()

# Optional statistical comparison: Spearman correlation between title keyword indicators and hourly rate.
from scipy.stats import spearmanr
rows=[]
for cat in keyword_groups:
    subset = rate_data.copy()
    indicator = subset['categories'].eq(cat).astype(int)
    if indicator.sum() >= 30 and (indicator == 0).sum() >= 30:
        rho, p = spearmanr(indicator, subset['hourly_mid'], nan_policy='omit')
        rows.append({'category':cat,'spearman_rho':rho,'p_value':p,'n_keyword':int(indicator.sum())})
keyword_stats = pd.DataFrame(rows).sort_values('spearman_rho',ascending=False)
display(keyword_stats.round(4))

## 4. Task 2 — Emerging job categories
Compare the first and second halves of the well-covered period, and calculate weekly growth. Because the early part of the file is sparse, we use a data-driven start date for the dense period and show it explicitly.

In [ ]:
daily = df.set_index('date').sort_index()
daily_counts = daily.resample('D').size()
# Find first day after which a 7-day rolling count reaches at least 500 postings.
rolling7 = daily_counts.rolling(7, min_periods=7).sum()
valid = rolling7[rolling7 >= 500]
analysis_start = valid.index[0] if len(valid) else daily_counts.index.min()
print('Dense-period analysis starts:', analysis_start)
trend_df = exploded[exploded['date'] >= analysis_start].copy()
weekly_cat = trend_df.groupby(['week','categories']).size().rename('postings').reset_index()
weeks = sorted(weekly_cat['week'].unique())
if len(weeks) >= 2:
    split = weeks[len(weeks)//2]
    first = weekly_cat[weekly_cat['week'] < split].groupby('categories')['postings'].mean().rename('early_avg_weekly')
    second = weekly_cat[weekly_cat['week'] >= split].groupby('categories')['postings'].mean().rename('late_avg_weekly')
    emerging = pd.concat([first,second],axis=1).fillna(0)
    emerging['absolute_change'] = emerging['late_avg_weekly'] - emerging['early_avg_weekly']
    emerging['pct_change'] = np.where(emerging['early_avg_weekly']>0, 100*emerging['absolute_change']/emerging['early_avg_weekly'], np.nan)
    emerging = emerging.sort_values('absolute_change',ascending=False)
    display(emerging.head(15).round(2))
    topcats = emerging.head(8).index
    plot_data = weekly_cat[weekly_cat['categories'].isin(topcats)]
    plt.figure(figsize=(14,6)); sns.lineplot(data=plot_data,x='week',y='postings',hue='categories',marker='o')
    plt.title('Weekly posting trends for leading title categories'); plt.xticks(rotation=45); plt.tight_layout(); plt.show()
else:
    print('Not enough weekly periods to calculate growth reliably.')

## 5. Task 3 — Forecast high-demand job categories
A forecast needs enough regular time points. We aggregate daily counts into weekly counts and compare against a naive baseline using a chronological holdout. With only a short history, treat metrics as a prototype evaluation, not reliable long-term accuracy.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Use categories with sufficient total volume, then forecast the next week from lagged weekly counts.
cat_totals = trend_df.explode('categories').groupby('categories').size().sort_values(ascending=False)
selected_cats = cat_totals.head(8).index.tolist()
weekly_all = trend_df.explode('categories').groupby(['week','categories']).size().unstack(fill_value=0).sort_index()
weekly_all = weekly_all.reindex(columns=selected_cats, fill_value=0)
print('Selected categories:', selected_cats)

def evaluate_lag_forecast(series):
    s = series.astype(float).asfreq('W-MON', fill_value=0)
    if len(s) < 5:
        return None
    # Simple one-step lag baseline; time-ordered evaluation on last 2 weeks.
    test_n = min(2, max(1, len(s)//4))
    train, test = s.iloc[:-test_n], s.iloc[-test_n:]
    preds = []
    history = list(train.values)
    for actual in test.values:
        preds.append(history[-1])
        history.append(actual)
    return {'MAE':mean_absolute_error(test.values,preds),
            'RMSE':mean_squared_error(test.values,preds)**0.5,
            'test_weeks':len(test), 'last_week_count':float(s.iloc[-1]),
            'next_week_baseline_forecast':float(s.iloc[-1])}

forecast_rows=[]
for cat in selected_cats:
    result=evaluate_lag_forecast(weekly_all[cat])
    if result: forecast_rows.append({'category':cat,**result})
forecast_results=pd.DataFrame(forecast_rows).sort_values('next_week_baseline_forecast',ascending=False)
display(forecast_results.round(2))

if not weekly_all.empty:
    weekly_all[selected_cats].plot(figsize=(14,6),marker='o')
    plt.title('Weekly demand for selected categories'); plt.ylabel('Number of postings'); plt.tight_layout(); plt.show()
print('Interpretation: lower MAE/RMSE is better. This baseline repeats the previous week; do not claim high accuracy from this short dataset.')

## 6. Task 4 — Compare hourly rates across countries
The `country` field may represent the client or account location, not necessarily the freelancer location. Compare only records with valid hourly rates and show sample sizes. Rates should be verified as a common currency before publication.

In [ ]:
country_rates = df[(df['is_hourly']) & df['hourly_mid'].notna() & (~df['hourly_rate_suspect']) & (df['country']!='Unknown')].groupby('country').agg(
    job_count=('hourly_mid','size'), median_rate=('hourly_mid','median'), mean_rate=('hourly_mid','mean'),
    q25=('hourly_mid',lambda x:x.quantile(.25)), q75=('hourly_mid',lambda x:x.quantile(.75)))
country_rates = country_rates[country_rates['job_count'] >= 100].sort_values('median_rate',ascending=False)
display(country_rates.head(25).round(2))
plt.figure(figsize=(12,8)); country_rates.head(20).sort_values('median_rate').median_rate.plot(kind='barh')
plt.title('Median hourly rate by country (at least 100 records)'); plt.xlabel('Hourly rate'); plt.tight_layout(); plt.show()
# Interactive chart
import plotly.express as px
fig = px.bar(country_rates.reset_index().head(30), x='country', y='median_rate', hover_data=['job_count','mean_rate','q25','q75'], title='Median hourly rate by country')
fig.update_layout(xaxis_tickangle=-45); fig.show()

## 7. Task 5 — Personalized job recommendation engine
This content-based prototype uses TF-IDF over job titles and ranks jobs by similarity to the user profile. It is a recommendation ranking, not a model trained on click or hire outcomes. Try changing the profile below.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

jobs = df.drop_duplicates('link').copy().reset_index(drop=True)
jobs['title'] = jobs['title'].fillna('')
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2), min_df=2, max_features=30000)
title_matrix = vectorizer.fit_transform(jobs['title'])

def recommend_jobs(profile, hourly_only=False, country=None, top_n=10):
    """Return relevant postings based on text similarity and optional filters."""
    profile_vec = vectorizer.transform([profile])
    scores = cosine_similarity(profile_vec, title_matrix).ravel()
    result = jobs.copy()
    result['relevance_score'] = scores
    if hourly_only:
        result = result[result['is_hourly'] == True]
    if country and country.lower() != 'any':
        result = result[result['country'].str.lower() == country.lower()]
    return result.sort_values('relevance_score',ascending=False).head(top_n)[
        ['title','country','is_hourly','hourly_low','hourly_high','budget','published_date','link','relevance_score']]

# Example profile; replace with a real user's skills and interests.
display(recommend_jobs('Python data analyst SQL dashboard machine learning', hourly_only=False, top_n=10))

### Optional: turn recommendations into an interactive Colab form

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
profile_box = widgets.Textarea(value='Python, SQL, data analysis, machine learning', description='Skills:', layout=widgets.Layout(width='80%',height='80px'))
country_box = widgets.Dropdown(options=['Any'] + sorted(df['country'].dropna().unique().tolist()), value='Any', description='Country:')
hourly_box = widgets.Checkbox(value=False, description='Hourly jobs only')
button = widgets.Button(description='Recommend jobs', button_style='primary')
out = widgets.Output()
def on_click(_):
    with out:
        clear_output()
        display(recommend_jobs(profile_box.value, hourly_only=hourly_box.value, country=country_box.value, top_n=15))
button.on_click(on_click)
display(profile_box, country_box, hourly_box, button, out)

## 8. Task 6 — Track changes by month
The dataset spans only parts of several months, so monthly comparisons are incomplete at the edges. Weekly trends are usually more informative here.

In [ ]:
monthly = df.groupby('month').agg(
    postings=('title','size'), hourly_postings=('is_hourly','sum'), countries=('country','nunique'))
monthly['fixed_price_postings'] = monthly['postings'] - monthly['hourly_postings']
display(monthly)
monthly[['hourly_postings','fixed_price_postings']].plot(kind='bar',figsize=(12,5))
plt.title('Hourly versus fixed-price postings by month'); plt.ylabel('Postings'); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

# Monthly median rate, based only on available and non-suspect hourly records.
monthly_rates = df[(df['is_hourly']) & df['hourly_mid'].notna() & (~df['hourly_rate_suspect'])].groupby('month')['hourly_mid'].median()
display(monthly_rates.to_frame('median_hourly_rate'))
monthly_rates.plot(marker='o',figsize=(10,4),title='Median hourly rate by month'); plt.tight_layout(); plt.show()

## 9. Task 7 — Remote-work landscape
This dataset does not contain an explicit remote/hybrid/on-site field. We can only measure mentions in job titles; absence of a keyword does not mean a job is on-site or non-remote.

In [ ]:
remote_pattern = r'\b(remote|work from home|wfh|hybrid|anywhere|distributed team)\b'
df['remote_keyword_mentioned'] = df['title'].str.lower().str.contains(remote_pattern, regex=True, na=False)
remote_summary = df.groupby('month')['remote_keyword_mentioned'].agg(['sum','count','mean']).rename(columns={'sum':'remote_keyword_jobs','count':'all_jobs','mean':'share'})
remote_summary['share_pct'] = 100*remote_summary['share']
display(remote_summary.round(3))
remote_summary['share_pct'].plot(marker='o',figsize=(10,4),title='Share of titles mentioning remote-work terms'); plt.ylabel('% of postings'); plt.tight_layout(); plt.show()
display(df[df['remote_keyword_mentioned']][['title','country','published_date','link']].head(20))

## 10. Task 8 — Future job-market trends and scenarios
Use observed trends as scenarios rather than strong predictions. With this short dataset, long-term forecasts are not statistically dependable. The code below calculates recent category share and a simple linear trend for exploration.

In [ ]:
trend_rows=[]
for cat in selected_cats:
    s = weekly_all[cat].astype(float).reset_index(drop=True)
    x = np.arange(len(s))
    if len(s) >= 3:
        slope, intercept = np.polyfit(x, s.values, 1)
        next_value = max(0, intercept + slope*len(s))
        trend_rows.append({'category':cat,'observed_weekly_avg':s.mean(),'linear_slope_per_week':slope,'illustrative_next_week':next_value})
future_scenarios = pd.DataFrame(trend_rows).sort_values('illustrative_next_week',ascending=False)
display(future_scenarios.round(2))
print('Scenario guidance: report these as exploratory estimates, not confirmed predictions. Validate with more months/years of data before making hiring or career decisions.')

## 11. Export results for report and dashboard
Save cleaned data and the tables/charts used in your report.

In [ ]:
OUTPUT_DIR = '/content/job_market_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
df.to_csv(f'{OUTPUT_DIR}/cleaned_jobs.csv', index=False)
keyword_salary.to_csv(f'{OUTPUT_DIR}/keyword_salary_analysis.csv')
if 'emerging' in globals(): emerging.to_csv(f'{OUTPUT_DIR}/emerging_categories.csv')
if 'forecast_results' in globals(): forecast_results.to_csv(f'{OUTPUT_DIR}/demand_forecast_evaluation.csv', index=False)
country_rates.to_csv(f'{OUTPUT_DIR}/country_hourly_rates.csv')
monthly.to_csv(f'{OUTPUT_DIR}/monthly_market_trends.csv')
remote_summary.to_csv(f'{OUTPUT_DIR}/remote_keyword_trends.csv')
future_scenarios.to_csv(f'{OUTPUT_DIR}/future_scenarios.csv', index=False)
print('Saved results to', OUTPUT_DIR)


## 12. Conclusions template
Fill this section with values produced by your run:
1. The most common categories in this dataset were …
2. Categories with the highest median hourly rates (after filtering) were …
3. The fastest-growing categories between the selected periods were …
4. Country-level median rates varied, but sample sizes and currency assumptions should be considered.
5. Recommendations are based on title similarity and can be improved with skill tags, seniority, budgets, and user feedback.
6. Remote-work results measure title keywords only.
7. Forecasts are exploratory because the observed time range is short.